In [1]:
import numpy as np
import time
from scapy.all import sniff
from tensorflow.keras.models import load_model

# 1. Load the pre-trained model
ann_model = load_model("intrusion_ann_model.h5")
print("Model loaded successfully.")
print(f"Model expects {ann_model.input_shape[1]} features.")

# 2. Initialize flow tracking variables
flow = {
    "start_time": None,
    "end_time": None,
    "fwd_packets": 0,
    "bwd_packets": 0,
    "fwd_bytes": 0,
    "bwd_bytes": 0,
}

SRC_IP = None  # Will be set by the first packet

def process_packet(pkt):
    global flow, SRC_IP

    if not pkt.haslayer("IP"):
        return

    size = len(pkt)

    if flow["start_time"] is None:
        flow["start_time"] = time.time()
        SRC_IP = pkt["IP"].src

    flow["end_time"] = time.time()
    src = pkt["IP"].src

    if src == SRC_IP:
        flow["fwd_packets"] += 1
        flow["fwd_bytes"] += size
    else:
        flow["bwd_packets"] += 1
        flow["bwd_bytes"] += size

def compute_and_predict():
    # Calculate durations and averages
    duration = flow["end_time"] - flow["start_time"] if flow["start_time"] else 0
    total_packets = flow["fwd_packets"] + flow["bwd_packets"]
    total_bytes = flow["fwd_bytes"] + flow["bwd_bytes"]

    fwd_mean = flow["fwd_bytes"] / flow["fwd_packets"] if flow["fwd_packets"] > 0 else 0
    bwd_mean = flow["bwd_bytes"] / flow["bwd_packets"] if flow["bwd_packets"] > 0 else 0
    
    avg_packet = total_bytes / total_packets if total_packets > 0 else 0
    bytes_per_sec = total_bytes / duration if duration > 0 else 0
    packets_per_sec = total_packets / duration if duration > 0 else 0

    # This list must match the EXACT order and number of features the model was trained on
    features = [
        duration,
        flow["fwd_packets"],
        flow["bwd_packets"],
        flow["fwd_bytes"],
        flow["bwd_bytes"],
        fwd_mean,
        bwd_mean,
        bytes_per_sec,
        packets_per_sec,
        avg_packet
    ]

    # --- MODEL PREDICTION ---
    # Convert list to numpy array and reshape for the model (1, number_of_features)
    input_data = np.array([features])
    
    # Check if we need to pad or trim features to match model input (n)
    expected_n = ann_model.input_shape[1]
    if input_data.shape[1] != expected_n:
        print(f"Warning: Extracted {input_data.shape[1]} features, but model needs {expected_n}.")
        # Basic padding with zeros if necessary
        input_data = np.pad(input_data, ((0,0), (0, max(0, expected_n - input_data.shape[1]))))[:, :expected_n]

    prediction_prob = ann_model.predict(input_data, verbose=0)
    prediction = (prediction_prob > 0.5).astype(int)

    print("\n--- Detection Result ---")
    print(f"Inference Probability: {prediction_prob[0][0]:.4f}")
    
    if prediction[0][0] == 1:
        print("⚠ ALERT: Intrusion Detected!")
    else:
        print("✅ Status: Normal Traffic")

# --- EXECUTION ---
print("Starting live sniff (20 packets)...")
sniff(prn=process_packet, count=2000)

print("Processing complete. Running analysis...")
compute_and_predict()



Model loaded successfully.
Model expects 10 features.
Starting live sniff (20 packets)...
Processing complete. Running analysis...

--- Detection Result ---
Inference Probability: 0.0000
✅ Status: Normal Traffic


In [3]:
import psutil
import time
from scapy.all import get_if_addr, conf

def get_system_usage():
    # 1. CPU Usage
    cpu_percent = psutil.cpu_percent(interval=1)

    # 2. Memory (RAM) Usage
    memory = psutil.virtual_memory()
    mem_used = memory.percent

    # 3. Disk Usage
    disk = psutil.disk_usage('/')
    disk_used = disk.percent

    # 4. WiFi/Network Usage
    # This tracks bytes sent/received since the last check
    net_before = psutil.net_io_counters(pernic=True)
    time.sleep(1) # Gap to calculate rate
    net_after = psutil.net_io_counters(pernic=True)

    # Automatically detect your WiFi interface (common names: 'Wi-Fi', 'wlan0', 'en0')
    wifi_iface = None
    for iface in psutil.net_if_addrs():
        if "wifi" in iface.lower() or "wlan" in iface.lower():
            wifi_iface = iface
            break
    
    # Fallback to default if WiFi name isn't standard
    if not wifi_iface:
        wifi_iface = list(net_after.keys())[0]

    stats_before = net_before[wifi_iface]
    stats_after = net_after[wifi_iface]

    # Calculate Speed (KB/s)
    upload_speed = (stats_after.bytes_sent - stats_before.bytes_sent) / 1024
    download_speed = (stats_after.bytes_recv - stats_before.bytes_recv) / 1024

    # Print Results
    print("-" * 30)
    print(f"💻 CPU Usage:    {cpu_percent}%")
    print(f"🧠 RAM Usage:    {mem_used}%")
    print(f"💽 Disk Usage:   {disk_used}%")
    print(f"📡 Interface:    {wifi_iface}")
    print(f"   ⬆️ Upload:    {upload_speed:.2f} KB/s")
    print(f"   ⬇️ Download:  {download_speed:.2f} KB/s")
    print("-" * 30)

if __name__ == "__main__":
    try:
        print("Monitoring System... (Press Ctrl+C to stop)")
        while True:
            get_system_usage()
    except KeyboardInterrupt:
        print("\nStopped.")

Monitoring System... (Press Ctrl+C to stop)
------------------------------
💻 CPU Usage:    50.4%
🧠 RAM Usage:    91.5%
💽 Disk Usage:   67.5%
📡 Interface:    Ethernet 2
   ⬆️ Upload:    0.00 KB/s
   ⬇️ Download:  0.00 KB/s
------------------------------
------------------------------
💻 CPU Usage:    52.1%
🧠 RAM Usage:    91.5%
💽 Disk Usage:   67.5%
📡 Interface:    Ethernet 2
   ⬆️ Upload:    0.00 KB/s
   ⬇️ Download:  0.00 KB/s
------------------------------
------------------------------
💻 CPU Usage:    53.7%
🧠 RAM Usage:    91.4%
💽 Disk Usage:   67.5%
📡 Interface:    Ethernet 2
   ⬆️ Upload:    0.00 KB/s
   ⬇️ Download:  0.00 KB/s
------------------------------
------------------------------
💻 CPU Usage:    49.5%
🧠 RAM Usage:    90.2%
💽 Disk Usage:   67.5%
📡 Interface:    Ethernet 2
   ⬆️ Upload:    0.00 KB/s
   ⬇️ Download:  0.00 KB/s
------------------------------
------------------------------
💻 CPU Usage:    45.5%
🧠 RAM Usage:    90.0%
💽 Disk Usage:   67.5%
📡 Interface:    Etherne

In [ ]:
import os
import platform

def get_command_history(n):
    history = []
    system = platform.system()

    try:
        if system == "Windows":
            # Path for PowerShell history
            history_path = os.path.expanduser(r"~\AppData\Roaming\Microsoft\Windows\PowerShell\PSReadLine\ConsoleHost_history.txt")
        else:
            # Path for Bash or Zsh (Linux/macOS)
            # Tries .bash_history first, then .zsh_history
            history_path = os.path.expanduser("~/.bash_history")
            if not os.path.exists(history_path):
                history_path = os.path.expanduser("~/.zsh_history")

        if os.path.exists(history_path):
            with open(history_path, "rb") as f:
                # Read lines and decode, ignoring errors for weird binary characters
                lines = f.read().decode(errors="ignore").splitlines()
                # Get the last n commands
                history = lines[-n:]
        else:
            return f"Error: History file not found at {history_path}"

    except Exception as e:
        return f"An error occurred: {e}"

    return history

# --- INPUT ---
try:
    n_input = 50
    
    print(f"\n--- Last {n_input} Commands ---")
    commands = get_command_history(n_input)
    
    if isinstance(commands, list):
        for i, cmd in enumerate(commands, 1):
            # Clean up Zsh timestamps if present (e.g., ': 1710432000:0;command')
            clean_cmd = cmd.split(';')[-1] if ';' in cmd else cmd
            print(f"{i}: {clean_cmd}")
    else:
        print(commands)
        
except ValueError:
    print("Please enter a valid number.")


--- Last 5 Commands ---
1: & C:/Users/hridy/AppData/Local/Programs/Python/Python314/python.exe c:/Users/hridy/OneDrive/Desktop/hack/check.py
2: python
3: python check.py
4: pip install psutil
5: python usage.py
